# Installations

In [1]:
%load_ext autoreload
%autoreload 2

In [1]:
%pip install -q mlflow databricks-sdk optuna

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 122.7 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 95.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 67.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 62.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Setup

In [2]:
try:
  import google.colab
  RUN_ENV = 'colab'
except ImportError:
  RUN_ENV = 'local'

In [3]:
COLAB_KAGGLE_UTILS_PATH = '/content/drive/MyDrive/KaggleData'
LOCAL_KAGGLE_UTILS_PATH = '/home/sameera/Projects'
COMP = 's6e8'

In [4]:
import sys

if RUN_ENV == 'colab':
    sys.path.append(COLAB_KAGGLE_UTILS_PATH)
    dataPath = '/content/drive/MyDrive/KaggleData/' + COMP + '/'
    from google.colab import drive
    drive.mount('/content/drive')
    DATABRICK_CONFIG_PATH = '/content/drive/MyDrive/KaggleData/mlflow/databricks.json'
    KAGGLE_CONFIG_PATH = '/content/drive/MyDrive/KaggleData/kaggle.json'
else:
    sys.path.append(LOCAL_KAGGLE_UTILS_PATH)
    dataPath = '/home/sameera/Projects/Kaggle/data/KaggleData/' + COMP + '/'
    DATABRICK_CONFIG_PATH = '/home/sameera/Projects/Kaggle/databricks.json'
    KAGGLE_CONFIG_PATH = '/home/sameera/Projects/Kaggle/kaggle.json'

In [5]:
import pandas as pd

from imblearn.over_sampling import RandomOverSampler
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer

# # Models CPU
# if RUN_ENV == 'local':
#   from sklearn.ensemble import RandomForestClassifier

# if RUN_ENV == 'colab':
# #   from cuml.ensemble import RandomForestClassifier
#   import cupy as cp

from xgboost import XGBClassifier

# Tuning
import optuna
# from optuna.integration import LightGBMPruningCallback

import mlflow
import os
import json

# Metrics 
from sklearn.metrics import roc_auc_score

# Custom utilities
from kaggle_utils.data.preprocessor import PreprocessorFactory
from kaggle_utils.models.learner import Learner
# from kaggle_utils.models.gpu_tuner import GPUTuner

/home/sameera/miniconda3/envs/kaggle_comp_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
train = pd.read_csv(dataPath + 'train.csv', low_memory=False)
test = pd.read_csv(dataPath + 'test.csv', low_memory=False)
sample_submission = pd.read_csv(dataPath + 'sample_submission.csv', index_col='id', low_memory=False)

# Data Glance

In [7]:
train.shape, test.shape, sample_submission.shape

((691369, 14), (296302, 13), (296302, 1))

In [13]:
train.head()

,id,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,gender,stress_level,academic_work_impact,addicted_label
0,0,24.0,NaN,1.83,1.59,2.11,7.46,122.0,38.0,8.63,Male,Medium,No,1
1,1,19.0,5.97,1.08,NaN,3.03,8.22,76.0,19.0,NaN,Female,Medium,No,0
2,2,18.0,5.09,NaN,NaN,NaN,6.25,134.0,60.0,7.47,Female,Low,Yes,0
3,3,21.0,6.42,1.26,1.42,3.36,8.85,112.0,94.0,8.66,Other,Low,NaN,1
4,4,26.0,11.20,1.87,2.81,1.95,5.25,NaN,NaN,13.39,Female,Medium,No,1


In [14]:
test.head()

,id,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,gender,stress_level,academic_work_impact
0,691369,30.0,9.34,NaN,0.77,4.09,7.15,153.0,16.0,NaN,Other,Medium,Yes
1,691370,NaN,NaN,1.91,NaN,1.20,8.67,239.0,148.0,10.68,Female,High,No
2,691371,26.0,8.48,3.64,NaN,3.39,7.47,106.0,123.0,NaN,Male,NaN,No
3,691372,20.0,8.37,2.99,1.69,2.53,5.45,178.0,55.0,9.88,Male,High,Yes
4,691373,25.0,8.25,4.02,1.21,2.32,8.46,63.0,86.0,10.72,Male,Low,No


In [15]:
sample_submission.head()

,addicted_label
id,
691369,0.709424
691370,0.709424
691371,0.709424
691372,0.709424
691373,0.709424


In [10]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 691369 entries, 0 to 691368
Data columns (total 14 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       691369 non-null  int64  
 1   age                      662440 non-null  float64
 2   daily_screen_time_hours  595515 non-null  float64
 3   social_media_hours       557374 non-null  float64
 4   gaming_hours             564548 non-null  float64
 5   work_study_hours         639851 non-null  float64
 6   sleep_hours              646889 non-null  float64
 7   notifications_per_day    623785 non-null  float64
 8   app_opens_per_day        610659 non-null  float64
 9   weekend_screen_time      579306 non-null  float64
 10  gender                   662335 non-null  object 
 11  stress_level             636221 non-null  object 
 12  academic_work_impact     647145 non-null  object 
 13  addicted_label           691369 non-null  int64  
dtypes: f

In [11]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 296302 entries, 0 to 296301
Data columns (total 13 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   id                       296302 non-null  int64  
 1   age                      279164 non-null  float64
 2   daily_screen_time_hours  263514 non-null  float64
 3   social_media_hours       248905 non-null  float64
 4   gaming_hours             236882 non-null  float64
 5   work_study_hours         268525 non-null  float64
 6   sleep_hours              273847 non-null  float64
 7   notifications_per_day    262081 non-null  float64
 8   app_opens_per_day        270597 non-null  float64
 9   weekend_screen_time      245605 non-null  float64
 10  gender                   282090 non-null  object 
 11  stress_level             276676 non-null  object 
 12  academic_work_impact     270581 non-null  object 
dtypes: float64(9), int64(1), object(3)
memory usage: 29.4+ MB


# Features

In [8]:
conts = ['age', 'daily_screen_time_hours', 'social_media_hours','gaming_hours', 'work_study_hours', 'sleep_hours','notifications_per_day', 'app_opens_per_day', 'weekend_screen_time',]
cats = ['gender', 'stress_level', 'academic_work_impact']

target = 'addicted_label'

# Preprocessing

## Encoding

In [9]:
catOrders = {
  "stress_level": ['Low', 'Medium', 'High']
}

In [9]:
# le  = LabelEncoder()
# train[target] = le.fit_transform(train[target])

In [10]:
feature_cols = conts + cats

### Testing Preprocessor

In [26]:
# Create the factory once
factory = PreprocessorFactory(conts, cats, catOrders)
# selected_features = ['sleep_duration', 'stress_level', 'physical_activity_level', 'bmi']

dynamic_preprocessor = factory.build(feature_cols)

In [27]:
factory.show_preprocessor()

ColumnTransformer(transformers=[('continuous', 'passthrough',
                                 ['age', 'daily_screen_time_hours',
                                  'social_media_hours', 'gaming_hours',
                                  'work_study_hours', 'sleep_hours',
                                  'notifications_per_day', 'app_opens_per_day',
                                  'weekend_screen_time']),
                                ('nominal',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value='Missing',
                                                                strategy='constant')),
                                                 ('encoder',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1))]),
                                 ['gender', 'academic_work_impa

In [18]:
t = dynamic_preprocessor.fit_transform(train[feature_cols])
print(t.head())

   continuous__age  continuous__daily_screen_time_hours  \
0             24.0                            999999.00   
1             19.0                                 5.97   
2             18.0                                 5.09   
3             21.0                                 6.42   
4             26.0                                11.20   

   continuous__social_media_hours  continuous__gaming_hours  \
0                            1.83                      1.59   
1                            1.08                 999999.00   
2                       999999.00                 999999.00   
3                            1.26                      1.42   
4                            1.87                      2.81   

   continuous__work_study_hours  continuous__sleep_hours  \
0                          2.11                     7.46   
1                          3.03                     8.22   
2                     999999.00                     6.25   
3                         

# Modelling

## setup MLflow

In [11]:
with open(DATABRICK_CONFIG_PATH, 'r') as f:
  config = json.load(f)

os.environ["DATABRICKS_HOST"] = config.get("DATABRICKS_HOST", "")
os.environ["DATABRICKS_TOKEN"] = config.get("DATABRICKS_TOKEN", "")
os.environ["MLFLOW_TRACKING_URI"] = "databricks"

In [12]:
user_email = "sameeranc11@gmail.com"
experiment_path = f"/Users/{user_email}/Kaggle_{COMP}_Comp"
# mlflow.sklearn.autolog(log_models=False) #removing this because  it's creating conflicts with manual logging.
mlflow.set_experiment(experiment_path)

If you are using MLflow Tracing, you can migrate your traces to Unity Catalog for unlimited storage, fine-grained access controls, and queryability from notebooks, SQL, and dashboards. Learn more: https://docs.databricks.com/aws/en/mlflow3/genai/tracing/migrate-traces-to-uc


<Experiment: artifact_location='dbfs:/databricks/mlflow-tracking/2419743223058656', creation_time=1785915213287, effective_trace_archival_retention=None, experiment_id='2419743223058656', last_update_time=1787122591423, lifecycle_stage='active', name='/Users/sameeranc11@gmail.com/Kaggle_s6e8_Comp', tags={'mlflow.experiment.sourceName': '/Users/sameeranc11@gmail.com/Kaggle_s6e8_Comp',
 'mlflow.experimentKind': 'custom_model_development',
 'mlflow.experimentType': 'MLFLOW_EXPERIMENT',
 'mlflow.ownerEmail': 'sameeranc11@gmail.com',
 'mlflow.ownerId': '75150582631674'}, trace_location=None, workspace='default'>

## setup kaggle

In [13]:
with open(KAGGLE_CONFIG_PATH, 'r') as f:
  config = json.load(f)

os.environ["KAGGLE_API_TOKEN"] = config.get("KAGGLE_API_TOKEN", "")

In [14]:
def submit_preds(model, target, fileName, le=None):
  final_string_predictions = le.inverse_transform(model.final_test_predictions) if le is not None else model.final_test_predictions
  sample_submission[target] = final_string_predictions
  sample_submission.to_csv(dataPath + fileName)

## Basic XBG

In [22]:
factory = PreprocessorFactory(conts, cats, catOrders)
dynamic_preprocessor = factory.build(feature_cols)
sampler = RandomOverSampler(random_state=42)

In [24]:
%%time
xgb_model = Learner(train, test, target,XGBClassifier,feature_cols,note="XGB_Kaggle_Tuned",metric=roc_auc_score,preprocessor=dynamic_preprocessor,needs_proba=True, target_class_idx=1, sampler=sampler)
xgb_model.fit(params={'n_estimators': 1000,'learning_rate': 0.01,'random_state': 42,'enable_categorical': True,'max_bin': 1024,'lambda': 4.315056676630924,'alpha': 2.214941127024784, 'colsample_bytree': 0.18360565346067603, 'subsample': 0.8941761954081464,'max_depth': 6,'min_child_weight': 5}, plotting={"fi":True, "score_vs_trees":True, "tree_depth": True})

Training fold 1...
Fold 1 ==> OOF score: 0.94158
Training fold 2...
Fold 2 ==> OOF score: 0.94169
Training fold 3...
Fold 3 ==> OOF score: 0.94249
Training fold 4...
🏃 View run XGB_Kaggle_Tuned at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/2419743223058656/runs/2d8b7a087fc84d36bccb99f8f2d2fccd
🧪 View experiment at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/2419743223058656
CPU times: user 40min 57s, sys: 7.81 s, total: 41min 4s
Wall time: 6min 55s


KeyboardInterrupt: 

In [40]:
submit_preds(xgb_model, target, 'XGB_Baseline_ClassBalanced_Ordinal.csv')   

In [42]:
!kaggle competitions submit -c playground-series-s6e8 -f /home/sameera/Projects/Kaggle/data/KaggleData/s6e8/XGB_Baseline_ClassBalanced_Ordinal.csv -m "XGB_Baseline_ClassBalanced_Ordinal"

100%|█████████████████████████████████████| 4.99M/4.99M [00:02<00:00, 2.14MB/s]
Successfully submitted to Predicting Smartphone Addiction

## Training based on model importance

In [21]:
fi_features = ['daily_screen_time_hours', 'social_media_hours', 'weekend_screen_time', 'app_opens_per_day','gaming_hours', 'work_study_hours']

In [20]:
for i in range(3,len(fi_features)+1):
  selected_features = fi_features[:i]
  print(f"Training with features: {selected_features}")
  dynamic_preprocessor = factory.build(selected_features)
  sampler = RandomOverSampler(random_state=42)

  %time
  xgb_model = Learner(train, test, target,XGBClassifier,selected_features,note=f"XGB_FI_{i}_top_features",metric=roc_auc_score,preprocessor=dynamic_preprocessor,needs_proba=True, target_class_idx=1, sampler=sampler)
  xgb_model.fit(plotting={"fi":True, "score_vs_trees":True, "tree_depth": True})

  submit_preds(xgb_model, target, f'XGB_FI_{i}_ClassBalanced_Ordinal.csv')

Training with features: ['daily_screen_time_hours', 'social_media_hours', 'weekend_screen_time']
CPU times: user 5 μs, sys: 1e+03 ns, total: 6 μs
Wall time: 11 μs
Training fold 1...
Fold 1 ==> OOF score: 0.93481
Training fold 2...
Fold 2 ==> OOF score: 0.93571
Training fold 3...
Fold 3 ==> OOF score: 0.93664
Training fold 4...
Fold 4 ==> OOF score: 0.93645
Training fold 5...
Fold 5 ==> OOF score: 0.93613
CV Mean Score: 0.9359 | Std: 0.0006
---------------------------------------------------------------
🏃 View run XGB_FI_3_top_features at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/2419743223058656/runs/ab6d9a75dfbe41fcb1537fb98093ddf6
🧪 View experiment at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/2419743223058656
Training with features: ['daily_screen_time_hours', 'social_media_hours', 'weekend_screen_time', 'app_opens_per_day']
CPU times: user 4 μs, sys: 0 ns, total: 4 μs
Wall time: 7.87 μs
Training fold 1...
Fold 1 ==> OOF score: 0.94697
Traini

In [ ]:
!kaggle competitions submit -c playground-series-s6e8 -f /home/sameera/Projects/Kaggle/data/KaggleData/s6e8/XGB_FI_{i}_ClassBalanced_Ordinal.csv -m "XGB_FI_{i}_ClassBalanced_Ordinal"

# RF Tuning

In [16]:
def rf_param_space(trial):
    """Defines the hyperparameter space for RandomForestClassifier."""
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500, 25),
        'max_depth': trial.suggest_int('max_depth', 7, 35, 2),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 2, 15, 1),
        'max_features':trial.suggest_int('max_features',2,12,1)
    }
    return params

In [ ]:
# factory = PreprocessorFactory(conts, cats, catOrders)
# selected_features = ['sleep_duration', 'stress_level', 'physical_activity_level', 'bmi', 'exercise_duration', 'step_count', 'sleep_quality']
# dynamic_preprocessor = factory.build(selected_features)

In [18]:
rf_tuner_model = Learner(train, test, target, RandomForestClassifier, feature_cols, f'RF_Baseline_ClassBalanced_Ordinal', roc_auc_score, preprocessor=dynamic_preprocessor, needs_proba=True, target_class_idx=1, sampler=sampler)
rf_tuner = GPUTuner(rf_tuner_model)
best_params = rf_tuner.fast_gpu_tune(study_name='RF_GPU_Tuning', n_trials=20, get_params_func=rf_param_space, bkp_db_path='/content/drive/MyDrive/KaggleData/s6e8/rf_gpu_tuning_study.db')

[I 2026-08-06 09:23:13,274] A new study created in RDB with name: RF_GPU_Tuning


  0%|          | 0/20 [00:00<?, ?it/s]

🏃 View run Trial_0 at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/2419743223058656/runs/3bb2e0e0f498419884c968b7210bf00d
🧪 View experiment at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/2419743223058656
[I 2026-08-06 09:24:23,177] Trial 0 finished with value: 0.9403698932220834 and parameters: {'n_estimators': 125, 'max_depth': 21, 'min_samples_leaf': 3, 'max_features': 4}. Best is trial 0 with value: 0.9403698932220834.
🏃 View run Trial_1 at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/2419743223058656/runs/f8dbf80a12c4484a8b02f80adb95d6e8
🧪 View experiment at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/experiments/2419743223058656
[I 2026-08-06 09:25:28,324] Trial 1 finished with value: 0.9311408673201796 and parameters: {'n_estimators': 325, 'max_depth': 7, 'min_samples_leaf': 14, 'max_features': 8}. Best is trial 0 with value: 0.9403698932220834.
🏃 View run Trial_2 at: https://dbc-36754e4d-ad0b.cloud.databricks.com/ml/e

: 

: 

: 

In [2]:
!date

Thu Aug  6 09:38:48 AM UTC 2026


In [31]:
best_params

In [42]:
%%time
rf_model = Learner(train, test, target, RandomForestClassifier, selected_features, 'RF_Tuned_ClassBalanced_Ordinal_TopF', balanced_accuracy_score, preprocessor=dynamic_preprocessor)
rf_model.fit(trainingMode='standard', params={'n_estimators': 190,
 'max_depth': 10,
 'min_samples_leaf': 12,
 'class_weight': 'balanced'}, ploting={"fi":True, "score_vs_trees":True})

In [43]:
submit_preds(rf_model, target, 'RF_Tuned_ClassBalanced_Ordinal_TopF.csv')   

# RF Kaggle Utils Testing - Conts missing large value imputing

In [26]:
factory = PreprocessorFactory(conts, cats, catOrders)
dynamic_preprocessor = factory.build(feature_cols, cont_imputer=SimpleImputer(strategy='constant', fill_value=999999))

In [33]:
rf_model = Learner(train, test, target, RandomForestClassifier, feature_cols, 'RF_Base_Cont_LV_Imputed', balanced_accuracy_score, preprocessor=dynamic_preprocessor)
rf_model.fit(trainingMode='standard', params={'max_depth': 10,
 'class_weight': 'balanced'}, plotting={"fi":True, "score_vs_trees":True, "tree_depth":True})

In [34]:
submit_preds(rf_model, target, 'RF_Base_Cont_LV_Imputed.csv')   